# Optimizer generality: quiet overnight runner

Run **Setup**, **Environment check**, then **Launch/resume**. The worker runs in the background; refresh **Status** manually. Stop, resume, results and export are separate cells. No prior research run is modified.

Defaults: SmolLM2-135M base and Pythia-410M; three fresh seeds; four main optimizers; paired label controls; preliminary Shampoo/Muon hybrids. This is a **62-job, multi-session queue** with an 11.5-hour budget per launch, not a promise that all experiments finish in one night.

**Important existing-data result:** the latest completed validation-loss increase is a stronger forecast baseline than the previous-update projection on the three text seeds. See `existing_timing_results/REPORT.txt` and README for the precise timeline and limitations.


In [ ]:
from pathlib import Path
import json, sys, subprocess
import experiment as study

OUTPUT = Path.cwd() / "runs" / "optimizer_generality_v1"
SETTINGS = study.defaults(OUTPUT)
candidates = [Path("/home/ubuntu/4/env_qwen3/bin/python"), Path("/home/ubuntu/ml_env/bin/python"), Path(sys.executable)]
SETTINGS["python"] = str(next(p for p in candidates if p.is_file()))

# Existing settings win on notebook reload, so Launch/resume uses the same experiment.
saved = OUTPUT / "settings.json"
if saved.exists():
    SETTINGS = json.loads(saved.read_text())

# Before FIRST launch only, optional scope changes:
# SETTINGS["label_models"] = ["smollm2_135m", "pythia410m"]
# SETTINGS["appendix"] = False
# SETTINGS["label_controls"] = False
# Operational budgets CAN be changed on resume:
SETTINGS["hours"] = 11.5
# SETTINGS["max_output_gib"] = 120.0
print("Worker Python:", SETTINGS["python"])
print("Results:", SETTINGS["output"])


### Environment check
Uses your existing CUDA environment; no package installation or Torch replacement is performed. Both appendix optimizers are included in the package. Requires Python 3.10+, Torch, Transformers, datasets, NumPy and Matplotlib on the GPU machine.

In [ ]:
check = subprocess.run([SETTINGS["python"], "-c", "import torch, transformers, datasets, numpy, matplotlib; from transformers import LlamaForCausalLM, GPTNeoXForCausalLM; print('Torch', torch.__version__, '| Transformers', transformers.__version__); print('CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU detected')"], text=True, capture_output=True)
if check.returncode:
    raise RuntimeError(check.stdout + "\n" + check.stderr)
print(check.stdout.strip())


### Launch/resume
Returns immediately. Re-running this cell does not create duplicate workers. The current job continues from its latest atomic checkpoint after a stop or budget pause.

In [ ]:
_ = study.launch(SETTINGS)


### Status — rerun this cell whenever you want an update
Reading status does not update the worker's progress timestamp. `alive=True` only establishes that the worker holds its lock; it does not guarantee progress.

In [ ]:
print(json.dumps(study.status(SETTINGS), indent=2))


### Results
Prints current summary. The plot appears when the complete queue finishes. Individual numerical audit failures and acquisition failures remain in the results.

In [ ]:
study.show_results(SETTINGS)


### Export the current results
Creates a consistent SQLite snapshot without model checkpoints or tokenized training corpora. Keep the full run directory for resuming or replay.

In [ ]:
share = study.export(SETTINGS)
print(share)
try:
    from IPython.display import FileLink, display
    display(FileLink(str(Path(share).relative_to(Path.cwd()))))
except ValueError:
    pass


### Stop — only run this when you want to stop
After requesting stop, refresh Status until `alive=False`. Run Launch/resume to continue. Do not delete checkpoints to stop a run.

In [ ]:
# Uncomment to stop:
# print(study.stop(SETTINGS))


### Fresh restart — optional
Creates a new output directory without deleting the previous one. Stop the previous worker first. The returned settings contain the new directory.

In [ ]:
# Uncomment ONLY for a fresh experiment:
# SETTINGS = study.restart(SETTINGS)
# OUTPUT = Path(SETTINGS["output"])
# print(OUTPUT)


### Optional: inspect the completed existing-record timing analysis
This report ships in the ZIP. The worker also reruns the analysis into the new output directory. No old ZIP or GPU is needed to read the shipped report.

In [ ]:
print((Path.cwd() / "existing_timing_results" / "REPORT.txt").read_text())


### Optional CPU smoke run
Exercises both tiny model adapters, all four main optimizers, label controls, both appendix variants and numerical probes. This is a software check, not research evidence. Keep its settings separate from the real GPU run.

In [ ]:
# SMOKE = study.smoke_settings(Path.cwd() / "runs" / "software_smoke")
# SMOKE["python"] = SETTINGS["python"]
# _ = study.launch(SMOKE)
# print(json.dumps(study.status(SMOKE), indent=2))
